## **Notebook 02 — BGE-M3 Embedder**
- Download BGE-M3, encode text into dense + sparse vectors, store in pgvector, run similarity search.


In [1]:
import os, sys, json
import numpy as np
from dotenv import load_dotenv

load_dotenv()
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")

print("Imports OK")

Imports OK


In [2]:
from FlagEmbedding import BGEM3FlagModel

# Points to your local models/ folder
# First run: downloads from HuggingFace automatically
# Second run: loads from disk — fast
MODEL_PATH = "../models/bge-m3"

model = BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    use_fp16=False,      # False = CPU mode, fine for practice
    cache_dir=MODEL_PATH
)

print("BGE-M3 loaded OK")

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

BGE-M3 loaded OK


In [3]:
sentences = [
    "Employees must submit leave requests 7 days in advance.",
    "All invoices must be approved by the finance department.",
    "The company provides 15 days of annual paid leave.",
    "Sales targets are reviewed every quarter by management.",
    "Remote work is allowed up to 3 days per week.",
]

output = model.encode(
    sentences,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,  # skip ColBERT for now
    batch_size=12,
)

dense  = output["dense_vecs"]       # shape: (5, 1024)
sparse = output["lexical_weights"]  # list of dicts: token_id -> weight

print("Dense shape   :", dense.shape)
print("Dense[0] preview (first 5 dims):", dense[0][:5].round(4))
print()
print("Sparse[0] top 5 tokens by weight:")
top5 = sorted(sparse[0].items(), key=lambda x: x[1], reverse=True)[:5]
for token_id, weight in top5:
    print(f"  token {token_id:>6}  weight {weight:.4f}")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Dense shape   : (5, 1024)
Dense[0] preview (first 5 dims): [-0.0287  0.0164 -0.0332  0.0005 -0.0165]

Sparse[0] top 5 tokens by weight:
  token  31358  weight 0.3264
  token 129745  weight 0.2431
  token  50336  weight 0.2322
  token   8110  weight 0.2115
  token 137447  weight 0.2098


In [4]:
# Cosine similarity — manually so you understand what pgvector does internally
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("Similarity scores (1.0 = identical, 0.0 = unrelated):\n")

# Similar meaning — should be HIGH
score = cosine_similarity(dense[0], dense[2])
print(f"  'Leave requests 7 days advance'")
print(f"  'Company provides 15 days annual leave'")
print(f"  Score: {score:.4f}  <- HIGH expected (both about leave)\n")

# Unrelated — should be LOW
score2 = cosine_similarity(dense[0], dense[3])
print(f"  'Leave requests 7 days advance'")
print(f"  'Sales targets reviewed quarterly'")
print(f"  Score: {score2:.4f}  <- LOW expected (different topics)")

Similarity scores (1.0 = identical, 0.0 = unrelated):

  'Leave requests 7 days advance'
  'Company provides 15 days annual leave'
  Score: 0.6926  <- HIGH expected (both about leave)

  'Leave requests 7 days advance'
  'Sales targets reviewed quarterly'
  Score: 0.4760  <- LOW expected (different topics)


In [5]:
import psycopg2
from pgvector.psycopg2 import register_vector

conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)   # tells psycopg2 how to handle vector type
cur      = conn.cursor()

# Insert a fake document row first (foreign key requirement)
cur.execute("""
    INSERT INTO documents (id, filename, file_type, status)
    VALUES (gen_random_uuid(), 'hr_policy_test.pdf', 'pdf', 'ready')
    RETURNING id;
""")
doc_id = cur.fetchone()[0]
print("Document ID:", doc_id)

# Insert each sentence as a chunk with its embedding
for i, (sentence, vec) in enumerate(zip(sentences, dense)):
    metadata = {
        "filename"   : "hr_policy_test.pdf",
        "chunk_index": i,
        "chunk_type" : "test",
        "section"    : "hr_policy",
    }
    cur.execute("""
        INSERT INTO chunks
            (document_id, content, embedding, metadata)
        VALUES (%s, %s, %s, %s::jsonb)
    """, (doc_id, sentence, vec.tolist(), json.dumps(metadata)))

conn.commit()

# Verify
cur.execute("SELECT COUNT(*) FROM chunks;")
print("Total chunks in DB:", cur.fetchone()[0])

Document ID: 3ee9bc63-8192-4673-9c57-eddfbf5ea1a6
Total chunks in DB: 5


In [6]:
# Query: something about time off — uses different words than the stored sentences
query = "How many vacation days do employees get?"

query_output  = model.encode([query], return_dense=True)
query_vec     = query_output["dense_vecs"][0]

# pgvector cosine similarity search
# <=> operator = cosine distance (lower = more similar)
# 1 - distance = similarity score
cur.execute("""
    SELECT
        content,
        metadata,
        1 - (embedding <=> %s::vector) AS similarity
    FROM chunks
    ORDER BY embedding <=> %s::vector
    LIMIT 3;
""", (query_vec.tolist(), query_vec.tolist()))

results = cur.fetchall()

print(f'Query: "{query}"\n')
print("Top 3 results from pgvector:\n")
for rank, (content, meta, score) in enumerate(results, 1):
    print(f"  Rank {rank} | Score: {score:.4f}")
    print(f"  {content}")
    print()

Query: "How many vacation days do employees get?"

Top 3 results from pgvector:

  Rank 1 | Score: 0.7040
  The company provides 15 days of annual paid leave.

  Rank 2 | Score: 0.6449
  Employees must submit leave requests 7 days in advance.

  Rank 3 | Score: 0.6141
  Remote work is allowed up to 3 days per week.



In [7]:
# Filter by section BEFORE doing vector search
# Useful when an org has multiple doc types — HR, Finance, Sales
# and you only want to search within one

cur.execute("""
    SELECT
        content,
        1 - (embedding <=> %s::vector) AS similarity
    FROM chunks
    WHERE metadata->>'section' = 'hr_policy'
    ORDER BY embedding <=> %s::vector
    LIMIT 3;
""", (query_vec.tolist(), query_vec.tolist()))

results = cur.fetchall()
print("Filtered to section='hr_policy' only:\n")
for rank, (content, score) in enumerate(results, 1):
    print(f"  Rank {rank} | Score: {score:.4f} | {content}")

Filtered to section='hr_policy' only:

  Rank 1 | Score: 0.7040 | The company provides 15 days of annual paid leave.
  Rank 2 | Score: 0.6449 | Employees must submit leave requests 7 days in advance.
  Rank 3 | Score: 0.6141 | Remote work is allowed up to 3 days per week.


In [ ]:
conn.close()

print("=" * 45)
print("STEP 3 COMPLETE")
print("=" * 45)
print("  BGE-M3 loads and encodes text     : OK")
print("  Dense vectors stored in pgvector  : OK")
print("  Cosine similarity search works    : OK")
print("  Metadata filtered search works    : OK")
print()
print("Key insight: query 'vacation days' matched")
print("'annual paid leave' — zero shared words.")
print("Semantic search working correctly.")
print()
print("READY FOR STEP 4 — Document Parser")

STEP 3 COMPLETE
  BGE-M3 loads and encodes text     : OK
  Dense vectors stored in pgvector  : OK
  Cosine similarity search works    : OK
  Metadata filtered search works    : OK

Key insight: query 'vacation days' matched
'annual paid leave' — zero shared words.
Semantic search working correctly.

READY FOR STEP 4 — Document Parser


: 